# Facial Expression Classification (FER)

This notebook adapts the supplied introductory CNN notebook to the uploaded FER dataset. It maps alternate emotion folder names into seven canonical classes and evaluates on the provided test split.

In [20]:
import json
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from train_model import (CLASS_NAMES, collect_images, make_dataset, build_model)

DATA_ROOT = Path("../FER-56-images-per-class/FER-56-images-per-class/facial_expression_source")
IMAGE_SIZE = (64, 64)
BATCH_SIZE = 32


## 1. Load image paths and labels

The dataset has a `train/` and `test/` split. Alternate spellings such as `fear` and `fearful` are treated as one class.

In [21]:
# train_paths, train_labels = collect_images(DATA_ROOT / "FER_56_per_class/train")
# test_paths, test_labels = collect_images(DATA_ROOT / "FER_56_per_class/test")
# print("Training images:", len(train_paths))
# print("Test images:", len(test_paths))
# for i, name in enumerate(CLASS_NAMES):
#     print(f"{name}: train={(train_labels == i).sum()}, test={(test_labels == i).sum()}")
from pathlib import Path

DATA_ROOT = Path.cwd()

# Find the actual project directory
while not (DATA_ROOT / "FER_56_per_class").exists():
    if DATA_ROOT.parent == DATA_ROOT:
        raise FileNotFoundError("Could not find FER_56_per_class")
    DATA_ROOT = DATA_ROOT.parent

print("Dataset root:", DATA_ROOT)
print("Train folder:", (DATA_ROOT / "FER_56_per_class" / "train").exists())
print("Test folder:", (DATA_ROOT / "FER_56_per_class" / "test").exists())

Dataset root: c:\FER_facial_expression_project\fer_expression_project
Train folder: True
Test folder: True


## 2. Create training and validation datasets

In [22]:
# tr_p, val_p, tr_y, val_y = train_test_split(
#     train_paths, train_labels, test_size=0.20, random_state=42, stratify=train_labels
# )
# train_ds = make_dataset(tr_p, tr_y, training=True)
# val_ds = make_dataset(val_p, val_y)
# test_ds = make_dataset(test_paths, test_labels)
train_paths, train_labels = collect_images(
    DATA_ROOT / "FER_56_per_class" / "train"
)

test_paths, test_labels = collect_images(
    DATA_ROOT / "FER_56_per_class" / "test"
)

print("Training images:", len(train_paths))
print("Testing images:", len(test_paths))

for i, name in enumerate(CLASS_NAMES):
    print(
        f"{name}: "
        f"train={(train_labels == i).sum()}, "
        f"test={(test_labels == i).sum()}"
    )

Training images: 392
Testing images: 7178
angry: train=56, test=958
disgusted: train=56, test=111
fearful: train=56, test=1024
happy: train=56, test=1774
neutral: train=56, test=1233
sad: train=56, test=1247
surprised: train=56, test=831


## 3. Build and train the CNN

In [23]:
# weights = compute_class_weight(class_weight="balanced", classes=np.unique(tr_y), y=tr_y)
# class_weights = {int(k): float(v) for k, v in zip(np.unique(tr_y), weights)}
# model = build_model()
# model.summary()
# callbacks = [
#     tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
#     tf.keras.callbacks.ModelCheckpoint("facial_expression_model.keras", monitor="val_loss", save_best_only=True),
#     tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
# ]
# history = model.fit(train_ds, validation_data=val_ds, epochs=30, class_weight=class_weights, callbacks=callbacks)
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

# Calculate balanced class weights
classes = np.unique(tr_y)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=tr_y
)

class_weights = {
    int(k): float(v)
    for k, v in zip(classes, weights)
}

print("Class weights:", class_weights)

# Build CNN model
model = build_model()

# Display model architecture
model.summary()

# Training callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "facial_expression_model.keras",
        monitor="val_loss",
        save_best_only=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2
    )
]

# Train model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weights,
    callbacks=callbacks
)

# Save final model
model.save("facial_expression_model.keras")

print("Training completed successfully!")

NameError: name 'tr_y' is not defined

## 4. Plot training history

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="Training")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(history.history["loss"], label="Training")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout()


## 5. Evaluate on the held-out test set

In [ ]:
print(model.evaluate(test_ds, return_dict=True))
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)
print(classification_report(y_true, y_pred, labels=range(len(CLASS_NAMES)), target_names=CLASS_NAMES, zero_division=0))
cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(xticks_rotation=45, cmap="Blues")
plt.tight_layout()


## 6. Save model and class names

In [ ]:
model.save("facial_expression_model.keras")
with open("classes.json", "w", encoding="utf-8") as f:
    json.dump(CLASS_NAMES, f, indent=2)


## 7. Predict one image

In [ ]:
from PIL import Image
image_path = test_paths[0]
image = Image.open(image_path).convert("RGB").resize(IMAGE_SIZE)
x = np.asarray(image, dtype=np.float32)[None, ...]
scores = model.predict(x, verbose=0)[0]
print("Predicted:", CLASS_NAMES[int(scores.argmax())])
print(dict(zip(CLASS_NAMES, scores.round(4))))
